In [9]:
import os, sys
import psycopg2

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("__file__"), "..")))
from shared.embedding import embed
from _data_generator.pgvec_generator import generate


In [10]:
conn = psycopg2.connect(
    host="pgvector_snoql_lab",
    database="vectordb",
    user="user",
    password="password"
)

cur = conn.cursor()

In [4]:
DDL = """
-- EXTENSION
CREATE EXTENSION IF NOT EXISTS vector;

-- =========================
-- DROP (safe reset)
-- =========================
DROP TABLE IF EXISTS sales;
DROP TABLE IF EXISTS products;

-- =========================
-- PRODUCTS
-- =========================
CREATE TABLE products (
    id INT PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price NUMERIC(10,2),
    description TEXT,
    embedding VECTOR(384),
    created_at TIMESTAMP DEFAULT NOW()
);

-- =========================
-- SALES
-- =========================
CREATE TABLE sales (
    product_id INT REFERENCES products(id),
    sales_count INT,
    updated_at TIMESTAMP DEFAULT NOW(),
    PRIMARY KEY (product_id)
);

-- =========================
-- INDEXES
-- =========================
CREATE INDEX idx_products_embedding_cosine
ON products
USING ivfflat (embedding vector_cosine_ops)
WITH (lists = 100);

CREATE INDEX idx_products_category
ON products(category);
"""


def initialize_database():
    print("Connecting to database...")

    print("Initializing schema...")

    cur.execute(DDL)
    conn.commit()

    print("Database initialized successfully.")

    cur.execute("SELECT current_database();")
    print(cur.fetchone())

In [5]:
initialize_database()

Connecting to database...
Initializing schema...
Database initialized successfully.
('vectordb',)


In [6]:
generate()

Connecting to DB...
Generating 5000 products...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Inserted batch 200/5000
Inserted batch 400/5000
Inserted batch 600/5000
Inserted batch 800/5000
Inserted batch 1000/5000
Inserted batch 1200/5000
Inserted batch 1400/5000
Inserted batch 1600/5000
Inserted batch 1800/5000
Inserted batch 2000/5000
Inserted batch 2200/5000
Inserted batch 2400/5000
Inserted batch 2600/5000
Inserted batch 2800/5000
Inserted batch 3000/5000
Inserted batch 3200/5000
Inserted batch 3400/5000
Inserted batch 3600/5000
Inserted batch 3800/5000
Inserted batch 4000/5000
Inserted batch 4200/5000
Inserted batch 4400/5000
Inserted batch 4600/5000
Inserted batch 4800/5000
Inserted batch 5000/5000
Generating sales data...
DONE


In [ ]:
query = "electronics product"
query_vec = embed([query])[0].tolist()

cur.execute(
    """
    SELECT name, category, price
    FROM products
    WHERE category = 'electronics'
    -- ORDER BY embedding <=> %s::vector
    LIMIT 5
    """, (query_vec,)
)

results = cur.fetchall()

for r in results:
    print(">>>", r)

>>> ('Bluetooth Headphones 328', 'electronics', Decimal('162.00'))
>>> ('High Performance Headphones 130', 'electronics', Decimal('115.00'))
>>> ('High Performance Speaker 384', 'electronics', Decimal('179.00'))
>>> ('Wireless Smartphone 400', 'electronics', Decimal('869.00'))
>>> ('Noise Cancelling Speaker 573', 'electronics', Decimal('408.00'))


In [ ]:
# cur.execute("""
# DROP TABLE IF EXISTS sales;
# DROP TABLE IF EXISTS products;
# DROP EXTENSION IF EXISTS vector;
# """)
# conn.commit()
# print("Tables dropped")

In [8]:
cur.close()
conn.close()